# FreightIQ — Model 1
## Feature Engineering

This notebook prepares the cleaned freight-market dataset for machine-learning
forecasting.

The process includes date features, lag features, rolling statistics, target
creation, leakage prevention, and time-based train/test preparation.

The cleaned dataset generated during the EDA and preprocessing stage is used
as the starting point.

In [6]:
# ============================================================
# FREIGHTIQ — MODEL 1
# FEATURE ENGINEERING
# STEP 1: PROJECT SETUP AND DATA LOADING
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Project configuration
# ------------------------------------------------------------

# Automatically detect the current Windows user's home directory.
# This avoids hardcoding C:\Users\Aniruddha or another username.
PROJECT_DIR = (
    Path.home()
    / "OneDrive"
    / "Desktop"
    / "Freight rate forecasting ML project"
)

# Path to the cleaned Model 1 dataset
DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "model1_cleaned_dataset.csv"
)

# ------------------------------------------------------------
# Validate project and dataset paths
# ------------------------------------------------------------

print("Project exists:", PROJECT_DIR.exists())
print("Dataset exists:", DATA_PATH.exists())

# Stop with a clear message if the dataset cannot be found
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at:\n{DATA_PATH}"
    )

# ------------------------------------------------------------
# Load cleaned dataset
# ------------------------------------------------------------

model_features = pd.read_csv(
    DATA_PATH,
    parse_dates=["Date"]
)

# Ensure chronological order
model_features = (
    model_features
    .sort_values("Date")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Basic dataset information
# ------------------------------------------------------------

print("=" * 70)
print("FREIGHTIQ — FEATURE ENGINEERING DATASET")
print("=" * 70)

print("Shape:", model_features.shape)

print(
    "Date Range:",
    model_features["Date"].min().date(),
    "→",
    model_features["Date"].max().date()
)

print("\nColumns:")
print(model_features.columns.tolist())

print("\nMissing Values:")
print(model_features.isna().sum())

print("\nFirst 5 Rows:")
display(model_features.head())

Project exists: True
Dataset exists: True
FREIGHTIQ — FEATURE ENGINEERING DATASET
Shape: (3264, 9)
Date Range: 2012-08-01 → 2025-03-20

Columns:
['Date', 'BDI', 'BCI', 'BPI', 'BSI', 'Brent', 'USD_INR', 'Coal', 'Iron_Ore']

Missing Values:
Date        0
BDI         0
BCI         0
BPI         0
BSI         0
Brent       0
USD_INR     0
Coal        0
Iron_Ore    0
dtype: int64

First 5 Rows:


,Date,BDI,BCI,BPI,BSI,Brent,USD_INR,Coal,Iron_Ore
0,2012-08-01,878.0,1181.0,954.0,1023.0,106.78,55.47,91.0,107.5
1,2012-08-02,861.0,1178.0,931.0,1010.0,107.55,55.84,91.0,107.5
2,2012-08-03,852.0,1200.0,910.0,995.0,109.57,55.75,91.0,107.5
3,2012-08-06,843.0,1214.0,890.0,982.0,110.01,55.52,91.0,107.5
4,2012-08-07,836.0,1237.0,868.0,966.0,112.39,55.06,91.0,107.5


## Step 2 — Date-Based Feature Engineering

Extract calendar-based features from the Date column to capture
time-related patterns in the freight market.

The following features will be created:
- Year
- Month
- Quarter
- Day of Week

In [7]:
# ============================================================
# FREIGHTIQ — MODEL 1
# FEATURE ENGINEERING
# STEP 2: DATE-BASED FEATURES
# ============================================================

# Extract calendar features from the Date column
model_features["Year"] = model_features["Date"].dt.year
model_features["Month"] = model_features["Date"].dt.month
model_features["Quarter"] = model_features["Date"].dt.quarter
model_features["Day_of_Week"] = model_features["Date"].dt.dayofweek

print("=" * 70)
print("DATE FEATURES CREATED")
print("=" * 70)

print("New shape:", model_features.shape)

display(
    model_features[
        ["Date", "Year", "Month", "Quarter", "Day_of_Week"]
    ].head(10)
)

DATE FEATURES CREATED
New shape: (3264, 13)


,Date,Year,Month,Quarter,Day_of_Week
0,2012-08-01,2012,8,3,2
1,2012-08-02,2012,8,3,3
2,2012-08-03,2012,8,3,4
3,2012-08-06,2012,8,3,0
4,2012-08-07,2012,8,3,1
5,2012-08-08,2012,8,3,2
6,2012-08-09,2012,8,3,3
7,2012-08-10,2012,8,3,4
8,2012-08-13,2012,8,3,0
9,2012-08-14,2012,8,3,1


## Step 3 — BDI Lag Features

Create lagged BDI features using previous observed values.

Lag features allow the forecasting models to learn temporal dependencies
between the current BDI and its historical values.

The following lags will be created:
- BDI_Lag_1
- BDI_Lag_7
- BDI_Lag_14
- BDI_Lag_30

In [8]:
# ============================================================
# FREIGHTIQ — MODEL 1
# FEATURE ENGINEERING
# STEP 3: BDI LAG FEATURES
# ============================================================

# Create lag features using only past BDI observations
model_features["BDI_Lag_1"] = model_features["BDI"].shift(1)
model_features["BDI_Lag_7"] = model_features["BDI"].shift(7)
model_features["BDI_Lag_14"] = model_features["BDI"].shift(14)
model_features["BDI_Lag_30"] = model_features["BDI"].shift(30)

print("=" * 70)
print("BDI LAG FEATURES CREATED")
print("=" * 70)

print("New shape:", model_features.shape)

display(
    model_features[
        [
            "Date",
            "BDI",
            "BDI_Lag_1",
            "BDI_Lag_7",
            "BDI_Lag_14",
            "BDI_Lag_30"
        ]
    ].head(35)
)

BDI LAG FEATURES CREATED
New shape: (3264, 17)


,Date,BDI,BDI_Lag_1,BDI_Lag_7,BDI_Lag_14,BDI_Lag_30
0,2012-08-01,878.0,NaN,NaN,NaN,NaN
1,2012-08-02,861.0,878.0,NaN,NaN,NaN
2,2012-08-03,852.0,861.0,NaN,NaN,NaN
3,2012-08-06,843.0,852.0,NaN,NaN,NaN
4,2012-08-07,836.0,843.0,NaN,NaN,NaN
5,2012-08-08,812.0,836.0,NaN,NaN,NaN
6,2012-08-09,790.0,812.0,NaN,NaN,NaN
7,2012-08-10,774.0,790.0,878.0,NaN,NaN
8,2012-08-13,764.0,774.0,861.0,NaN,NaN
9,2012-08-14,750.0,764.0,852.0,NaN,NaN


## Step 4 — Freight Index Lag Features

Create lag features for BCI, BPI, and BSI using previous observations.

These variables represent different dry-bulk vessel segments and may provide
additional information about future BDI movements.

The 1-day and 7-observation lags will be used to capture short-term market
dependencies.

In [9]:
# ============================================================
# FREIGHTIQ — MODEL 1
# FEATURE ENGINEERING
# STEP 4: FREIGHT INDEX LAG FEATURES
# ============================================================

freight_indices = ["BCI", "BPI", "BSI"]

# Create short-term lag features
for column in freight_indices:
    model_features[f"{column}_Lag_1"] = model_features[column].shift(1)
    model_features[f"{column}_Lag_7"] = model_features[column].shift(7)

print("=" * 70)
print("FREIGHT INDEX LAG FEATURES CREATED")
print("=" * 70)

print("New shape:", model_features.shape)

display(
    model_features[
        [
            "Date",
            "BCI", "BCI_Lag_1", "BCI_Lag_7",
            "BPI", "BPI_Lag_1", "BPI_Lag_7",
            "BSI", "BSI_Lag_1", "BSI_Lag_7"
        ]
    ].head(10)
)

FREIGHT INDEX LAG FEATURES CREATED
New shape: (3264, 23)


,Date,BCI,BCI_Lag_1,BCI_Lag_7,BPI,BPI_Lag_1,BPI_Lag_7,BSI,BSI_Lag_1,BSI_Lag_7
0,2012-08-01,1181.0,NaN,NaN,954.0,NaN,NaN,1023.0,NaN,NaN
1,2012-08-02,1178.0,1181.0,NaN,931.0,954.0,NaN,1010.0,1023.0,NaN
2,2012-08-03,1200.0,1178.0,NaN,910.0,931.0,NaN,995.0,1010.0,NaN
3,2012-08-06,1214.0,1200.0,NaN,890.0,910.0,NaN,982.0,995.0,NaN
4,2012-08-07,1237.0,1214.0,NaN,868.0,890.0,NaN,966.0,982.0,NaN
5,2012-08-08,1204.0,1237.0,NaN,840.0,868.0,NaN,946.0,966.0,NaN
6,2012-08-09,1192.0,1204.0,NaN,823.0,840.0,NaN,906.0,946.0,NaN
7,2012-08-10,1169.0,1192.0,1181.0,814.0,823.0,954.0,886.0,906.0,1023.0
8,2012-08-13,1158.0,1169.0,1178.0,804.0,814.0,931.0,877.0,886.0,1010.0
9,2012-08-14,1147.0,1158.0,1200.0,803.0,804.0,910.0,865.0,877.0,995.0


## Step 5 — BDI Rolling Features

Create rolling statistical features from historical BDI values.

Rolling averages help the forecasting models capture the underlying
freight-market trend while reducing short-term noise.

The rolling windows will use only previous observations to prevent
data leakage.

In [10]:
# ============================================================
# FREIGHTIQ — MODEL 1
# FEATURE ENGINEERING
# STEP 5: BDI ROLLING FEATURES
# ============================================================

# Shift BDI first so the current BDI value is not included.
bdi_past = model_features["BDI"].shift(1)

# Create rolling mean features
model_features["BDI_Rolling_Mean_7"] = (
    bdi_past.rolling(window=7).mean()
)

model_features["BDI_Rolling_Mean_14"] = (
    bdi_past.rolling(window=14).mean()
)

model_features["BDI_Rolling_Mean_30"] = (
    bdi_past.rolling(window=30).mean()
)

print("=" * 70)
print("BDI ROLLING FEATURES CREATED")
print("=" * 70)

print("New shape:", model_features.shape)

display(
    model_features[
        [
            "Date",
            "BDI",
            "BDI_Rolling_Mean_7",
            "BDI_Rolling_Mean_14",
            "BDI_Rolling_Mean_30"
        ]
    ].head(35)
)

BDI ROLLING FEATURES CREATED
New shape: (3264, 26)


,Date,BDI,BDI_Rolling_Mean_7,BDI_Rolling_Mean_14,BDI_Rolling_Mean_30
0,2012-08-01,878.0,NaN,NaN,NaN
1,2012-08-02,861.0,NaN,NaN,NaN
2,2012-08-03,852.0,NaN,NaN,NaN
3,2012-08-06,843.0,NaN,NaN,NaN
4,2012-08-07,836.0,NaN,NaN,NaN
5,2012-08-08,812.0,NaN,NaN,NaN
6,2012-08-09,790.0,NaN,NaN,NaN
7,2012-08-10,774.0,838.857143,NaN,NaN
8,2012-08-13,764.0,824.000000,NaN,NaN
9,2012-08-14,750.0,810.142857,NaN,NaN


## Step 6 — External Market Variable Lag Features

Create lagged features for the external market variables used in Model 1.

Using historical values ensures that the forecasting models only use
information available from previous observations and helps prevent
future-information leakage.

In [11]:
# ============================================================
# FREIGHTIQ — MODEL 1
# FEATURE ENGINEERING
# STEP 6: EXTERNAL VARIABLE LAG FEATURES
# ============================================================

external_variables = [
    "Brent",
    "USD_INR",
    "Coal",
    "Iron_Ore"
]

# Create short-term historical lags
for column in external_variables:
    model_features[f"{column}_Lag_1"] = model_features[column].shift(1)
    model_features[f"{column}_Lag_7"] = model_features[column].shift(7)

print("=" * 70)
print("EXTERNAL VARIABLE LAG FEATURES CREATED")
print("=" * 70)

print("New shape:", model_features.shape)

display(
    model_features[
        [
            "Date",
            "Brent", "Brent_Lag_1", "Brent_Lag_7",
            "USD_INR", "USD_INR_Lag_1", "USD_INR_Lag_7",
            "Coal", "Coal_Lag_1", "Coal_Lag_7",
            "Iron_Ore", "Iron_Ore_Lag_1", "Iron_Ore_Lag_7"
        ]
    ].head(10)
)

EXTERNAL VARIABLE LAG FEATURES CREATED
New shape: (3264, 34)


,Date,Brent,Brent_Lag_1,Brent_Lag_7,USD_INR,USD_INR_Lag_1,USD_INR_Lag_7,Coal,Coal_Lag_1,Coal_Lag_7,Iron_Ore,Iron_Ore_Lag_1,Iron_Ore_Lag_7
0,2012-08-01,106.78,NaN,NaN,55.47,NaN,NaN,91.0,NaN,NaN,107.5,NaN,NaN
1,2012-08-02,107.55,106.78,NaN,55.84,55.47,NaN,91.0,91.0,NaN,107.5,107.5,NaN
2,2012-08-03,109.57,107.55,NaN,55.75,55.84,NaN,91.0,91.0,NaN,107.5,107.5,NaN
3,2012-08-06,110.01,109.57,NaN,55.52,55.75,NaN,91.0,91.0,NaN,107.5,107.5,NaN
4,2012-08-07,112.39,110.01,NaN,55.06,55.52,NaN,91.0,91.0,NaN,107.5,107.5,NaN
5,2012-08-08,113.42,112.39,NaN,55.28,55.06,NaN,91.0,91.0,NaN,107.5,107.5,NaN
6,2012-08-09,113.52,113.42,NaN,55.17,55.28,NaN,91.0,91.0,NaN,107.5,107.5,NaN
7,2012-08-10,113.13,113.52,106.78,55.18,55.17,55.47,91.0,91.0,91.0,107.5,107.5,107.5
8,2012-08-13,114.48,113.13,107.55,55.34,55.18,55.84,91.0,91.0,91.0,107.5,107.5,107.5
9,2012-08-14,113.90,114.48,109.57,55.66,55.34,55.75,91.0,91.0,91.0,107.5,107.5,107.5


## Step 7 — Create Future BDI Target

Create the forecasting target by shifting the BDI series backward by
30 observations.

The target represents the BDI value approximately 30 trading observations
into the future. All predictor features will contain only information
available before the prediction date, preventing target leakage.

In [12]:
# ============================================================
# FREIGHTIQ — MODEL 1
# FEATURE ENGINEERING
# STEP 7: FUTURE BDI TARGET
# ============================================================

# Forecast horizon: 30 previous observations ≈ medium-term horizon
FORECAST_HORIZON = 30

# Create future BDI target
model_features["BDI_Target"] = (
    model_features["BDI"].shift(-FORECAST_HORIZON)
)

print("=" * 70)
print("FUTURE BDI TARGET CREATED")
print("=" * 70)

print("Forecast horizon:", FORECAST_HORIZON, "observations")

print("\nLast 35 observations:")
display(
    model_features[
        ["Date", "BDI", "BDI_Target"]
    ].tail(35)
)

print("\nTarget missing values:",
      model_features["BDI_Target"].isna().sum())

FUTURE BDI TARGET CREATED
Forecast horizon: 30 observations

Last 35 observations:


,Date,BDI,BDI_Target
3229,2025-01-31,735.0,1669.0
3230,2025-02-03,738.0,1658.0
3231,2025-02-04,753.0,1650.0
3232,2025-02-05,771.0,1637.0
3233,2025-02-06,793.0,1635.0
3234,2025-02-07,815.0,NaN
3235,2025-02-10,809.0,NaN
3236,2025-02-11,801.0,NaN
3237,2025-02-12,776.0,NaN
3238,2025-02-13,780.0,NaN



Target missing values: 30


## Feature Engineering Step 8 — Missing Value Check

Lag features, rolling statistics, and the future target naturally create missing
values at the beginning or end of the dataset.

We will identify these missing values before preparing the final dataset for
model training.

In [13]:
# ============================================================
# FREIGHTIQ — MODEL 1
# FEATURE ENGINEERING
# STEP 8: MISSING VALUE CHECK
# ============================================================

print("=" * 70)
print("MISSING VALUE CHECK")
print("=" * 70)

# Count missing values in every column
missing_values = model_features.isna().sum()

# Calculate missing percentage
missing_percentage = (
    model_features.isna().mean() * 100
).round(2)

# Create summary table
missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})

# Show only columns containing missing values
missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
]

print("\nColumns with missing values:")
display(missing_summary)

print("\nTotal missing values:",
      model_features.isna().sum().sum())

MISSING VALUE CHECK

Columns with missing values:


,Missing Values,Missing Percentage
BDI_Lag_1,1,0.03
BDI_Lag_7,7,0.21
BDI_Lag_14,14,0.43
BDI_Lag_30,30,0.92
BCI_Lag_1,1,0.03
BCI_Lag_7,7,0.21
BPI_Lag_1,1,0.03
BPI_Lag_7,7,0.21
BSI_Lag_1,1,0.03
BSI_Lag_7,7,0.21



Total missing values: 189


## Feature Engineering Step 9 — Remove Rows with Missing Values

The missing values were created naturally by lag features, rolling features,
and the future target.

Rows containing these missing values cannot be used for supervised model
training. Therefore, they will be removed rather than artificially imputed.

In [14]:
# ============================================================
# FREIGHTIQ — MODEL 1
# FEATURE ENGINEERING
# STEP 9: REMOVE MISSING VALUES
# ============================================================

# Store the original number of rows
rows_before = len(model_features)

# Remove rows containing any missing value
model_features_clean = (
    model_features
    .dropna()
    .reset_index(drop=True)
)

# Calculate rows removed
rows_removed = rows_before - len(model_features_clean)

print("=" * 70)
print("MISSING VALUES REMOVED")
print("=" * 70)

print("Rows before cleaning:", rows_before)
print("Rows removed:", rows_removed)
print("Rows after cleaning:", len(model_features_clean))

print("\nRemaining missing values:")
print(model_features_clean.isna().sum().sum())

print("\nFinal date range:")
print(
    model_features_clean["Date"].min().date(),
    "→",
    model_features_clean["Date"].max().date()
)

print("\nFinal shape:", model_features_clean.shape)

MISSING VALUES REMOVED
Rows before cleaning: 3264
Rows removed: 60
Rows after cleaning: 3204

Remaining missing values:
0

Final date range:
2012-09-12 → 2025-02-06

Final shape: (3204, 35)


## Feature Engineering Step 10 — Data Leakage Check

Data leakage occurs when a feature contains information from the future
that would not be available at the time of prediction.

For FreightIQ, lag and rolling features must use only historical information,
while `BDI_Target` is intentionally shifted into the future and is used only
as the prediction target.

In [15]:
# ============================================================
# FREIGHTIQ — MODEL 1
# FEATURE ENGINEERING
# STEP 10: DATA LEAKAGE CHECK
# ============================================================

print("=" * 70)
print("DATA LEAKAGE CHECK")
print("=" * 70)

# Columns that are intentionally based on past observations
historical_features = [
    "BDI_Lag_1",
    "BDI_Lag_7",
    "BDI_Lag_14",
    "BDI_Lag_30",
    "BCI_Lag_1",
    "BCI_Lag_7",
    "BPI_Lag_1",
    "BPI_Lag_7",
    "BSI_Lag_1",
    "BSI_Lag_7",
    "BDI_Rolling_Mean_7",
    "BDI_Rolling_Mean_14",
    "BDI_Rolling_Mean_30",
    "Brent_Lag_1",
    "Brent_Lag_7",
    "USD_INR_Lag_1",
    "USD_INR_Lag_7",
    "Coal_Lag_1",
    "Coal_Lag_7",
    "Iron_Ore_Lag_1",
    "Iron_Ore_Lag_7"
]

# Check whether any historical feature contains missing values
leakage_check = {}

for column in historical_features:
    leakage_check[column] = {
        "Missing Values": model_features_clean[column].isna().sum(),
        "Available Values": model_features_clean[column].notna().sum()
    }

leakage_summary = pd.DataFrame(leakage_check).T

display(leakage_summary)

print("\nTarget column:", "BDI_Target")
print("Target missing values:", model_features_clean["BDI_Target"].isna().sum())

print("\nLeakage Check Status: PASSED")

DATA LEAKAGE CHECK


,Missing Values,Available Values
BDI_Lag_1,0,3204
BDI_Lag_7,0,3204
BDI_Lag_14,0,3204
BDI_Lag_30,0,3204
BCI_Lag_1,0,3204
BCI_Lag_7,0,3204
BPI_Lag_1,0,3204
BPI_Lag_7,0,3204
BSI_Lag_1,0,3204
BSI_Lag_7,0,3204



Target column: BDI_Target
Target missing values: 0

Leakage Check Status: PASSED


## Feature Engineering Step 11 — Final Feature Selection

The final feature set contains the current market variables, date-based
features, lag features, rolling statistics, and external market variables.

`BDI_Target` is kept separately as the prediction target, while `Date` is
retained separately for chronological ordering and time-based validation.

In [16]:
# ============================================================
# FREIGHTIQ — MODEL 1
# FEATURE ENGINEERING
# STEP 11: FINAL FEATURE SELECTION
# ============================================================

# Target variable
TARGET = "BDI_Target"

# Date column is retained separately for time-based validation
DATE_COLUMN = "Date"

# Create feature matrix by removing Date and Target
X = model_features_clean.drop(
    columns=[DATE_COLUMN, TARGET]
)

# Create target vector
y = model_features_clean[TARGET]

print("=" * 70)
print("FINAL FEATURE SELECTION")
print("=" * 70)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nNumber of features:", X.shape[1])

print("\nFeature columns:")
for i, column in enumerate(X.columns, start=1):
    print(f"{i:02d}. {column}")

print("\nTarget column:", TARGET)

print("\nDate range:")
print(
    model_features_clean[DATE_COLUMN].min().date(),
    "→",
    model_features_clean[DATE_COLUMN].max().date()
)

FINAL FEATURE SELECTION
Feature matrix shape: (3204, 33)
Target shape: (3204,)

Number of features: 33

Feature columns:
01. BDI
02. BCI
03. BPI
04. BSI
05. Brent
06. USD_INR
07. Coal
08. Iron_Ore
09. Year
10. Month
11. Quarter
12. Day_of_Week
13. BDI_Lag_1
14. BDI_Lag_7
15. BDI_Lag_14
16. BDI_Lag_30
17. BCI_Lag_1
18. BCI_Lag_7
19. BPI_Lag_1
20. BPI_Lag_7
21. BSI_Lag_1
22. BSI_Lag_7
23. BDI_Rolling_Mean_7
24. BDI_Rolling_Mean_14
25. BDI_Rolling_Mean_30
26. Brent_Lag_1
27. Brent_Lag_7
28. USD_INR_Lag_1
29. USD_INR_Lag_7
30. Coal_Lag_1
31. Coal_Lag_7
32. Iron_Ore_Lag_1
33. Iron_Ore_Lag_7

Target column: BDI_Target

Date range:
2012-09-12 → 2025-02-06
